# Phase 6: Final Prediction System

Loads the tuned model + vectorizer from Phase 5 and wraps them in a reusable
function that classifies a brand-new, unseen job posting.

In [1]:
import re
import joblib

model = joblib.load("../app/fraud_model.joblib")
vectorizer = joblib.load("../app/tfidf_vectorizer.joblib")
print("Loaded tuned model and TF-IDF vectorizer from app/")

Loaded tuned model and TF-IDF vectorizer from app/


## Cleaning function

Must match the cleaning used during training (Phase 2) exactly, since the
vectorizer was fit on text cleaned this same way.

In [2]:
def clean_text(text: str) -> str:
    """Lowercase, strip HTML tags, URLs, placeholder tokens, punctuation, and digits noise."""
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"#url_\w+#|#email_\w+#|#phone_\w+#", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

## The prediction function

In [3]:
def predict_job_posting(title="", company_profile="", description="", requirements="", benefits=""):
    """Classify a new job posting as fraudulent or legitimate."""
    raw = " ".join([title, company_profile, description, requirements, benefits])
    cleaned = clean_text(raw)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0][1]
    label = "FRAUDULENT" if pred == 1 else "LEGITIMATE"
    return {"prediction": label, "fraud_probability": round(float(prob), 4)}

## Demo on two example postings

In [4]:
example_legit = predict_job_posting(
    title="Senior Data Analyst",
    company_profile="Acme Corp is a 200-person analytics consultancy founded in 2010, based in Chicago.",
    description="We are looking for a Senior Data Analyst to join our growing team, working with SQL and Python.",
    requirements="3+ years experience with SQL, Python, and data visualization tools. Bachelor's degree required.",
    benefits="Health insurance, 401k matching, flexible PTO."
)
example_fraud = predict_job_posting(
    title="Easy Data Entry - Work From Home - Earn $500/day!!!",
    company_profile="",
    description="No experience needed! Just fill out simple forms from home and earn cash daily. Immediate start.",
    requirements="No qualifications needed. Must have a bank account to receive payments.",
    benefits="Unlimited earning potential! Be your own boss!"
)
print("Example 1 (looks legitimate):", example_legit)
print("Example 2 (looks like a scam):", example_fraud)

Example 1 (looks legitimate): {'prediction': 'LEGITIMATE', 'fraud_probability': 0.0014}
Example 2 (looks like a scam): {'prediction': 'FRAUDULENT', 'fraud_probability': 0.996}


## Limitations & Real-World Considerations

- **Class imbalance**: only ~5% of postings are fraudulent; even with `class_weight="balanced"`,
  rare fraud patterns not seen in training will be missed.
- **Vocabulary/temporal drift**: the model learns word patterns from 2012–2014 postings;
  scammers adapt their language over time, so periodic retraining on fresh data would be
  needed in production.
- **Text-only bias**: the model leans heavily on textual cues; sophisticated scams that
  closely mimic legitimate corporate language (no obvious "red flag" words) can slip through.
- **No external verification**: the system doesn't check whether the company actually exists,
  cross-reference domains/emails, or verify salary claims against market rates.
- **False positives have a cost too**: overly aggressive fraud-flagging could hide legitimate
  postings (e.g., legitimate remote/entry-level jobs, which structurally resemble common scam patterns).
- **Language coverage**: the training data is English-only; postings in other languages
  would need separate handling.

## Conclusion

Even a relatively simple TF-IDF + Logistic Regression pipeline achieves strong fraud-detection
performance on this dataset (F1 = 0.80, ROC-AUC = 0.986) by picking up on genuine linguistic
red flags (vague company profiles, unrealistic pay claims, urgency language). It's a solid
baseline, but production use would pair it with metadata verification and human review for
borderline cases.